# 测试集结果评估(scButterfly)

In [1]:
import pickle
import numpy as np
import polars as pl
from deeptan.utils.metrics import (
    calculate_jsd,
    calculate_mse,
    calculate_pcc,
    ARI,
    NMI,
    kBET,
    ASW,
    AUROC,
    F1,
)

/home/wuch/miniforge3/envs/sc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import scanpy as sc

## Read true values

### RNA

In [27]:
preprocessed_data_rna = sc.read_h5ad("/mnt/hdd2/homext/wuch/xn2p/comparison/scButterfly/denovo_try_1_f3000/preprocessed_data/normalize_True_log1p_True_hvg_True_3000_RNA_processed_data.h5ad")
print(preprocessed_data_rna)

AnnData object with n_obs × n_vars = 4371 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type'
    var: 'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'


For the count matrices of scRNAseq profiles, we normalized the total count of each cell to have the same values equal to the median of total counts for cells before normalization, logarithmized the normalized values with an adding offset of one,

上面读取的数据是已经预处理的

In [ ]:
# if not isinstance(preprocessed_data_rna.X, np.ndarray):
#     preprocessed_data_rna.X = preprocessed_data_rna.X.toarray()

# _total = preprocessed_data_rna.X.sum(axis=1)
# _median = np.median(_total)
# _scale_coef = _median / _total
# preprocessed_data_rna.X = preprocessed_data_rna.X * _scale_coef[:, np.newaxis]
# preprocessed_data_rna.X = np.log1p(preprocessed_data_rna.X)

In [ ]:
# print(preprocessed_data_rna)
# print(preprocessed_data_rna.X)

### ATAC

In [28]:
preprocessed_data_atac = sc.read_h5ad("/mnt/hdd2/homext/wuch/xn2p/comparison/scButterfly/denovo_try_1_f3000/preprocessed_data/binarize_True_filter_True_fpeaks_0.005_tfidf_True_normalize_True_ATAC_processed_data.h5ad")
print(preprocessed_data_atac)

AnnData object with n_obs × n_vars = 4371 × 24317
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type'
    var: 'gene_ids', 'feature_types'


For the count matrices of scATAC-seq profiles, we binarized the matrices, filtered out the peaks activated in less than 0.5% of all cells, and performed term frequency-inverse document frequency (TF-IDF) transformation

In [11]:
import numpy as np
from sklearn.feature_extraction.text import TfidfTransformer

def process_scatac_seq(count_matrix, min_activation_rate=0.005):
    """
    Process the count matrix of scATAC-seq profiles by binarizing, filtering, and applying TF-IDF transformation.

    Parameters:
        count_matrix (np.ndarray): The input count matrix of shape (n_cells, n_peaks).
        min_activation_rate (float): Minimum activation rate to retain a peak (default is 0.5%).

    Returns:
        tfidf_matrix (np.ndarray): The processed TF-IDF transformed matrix.
    """
    # Step 1: Binarize the matrix (non-zero values become 1)
    binary_matrix = (count_matrix > 0).astype(int)

    # Step 2: Filter out peaks activated in less than min_activation_rate of all cells
    n_cells = binary_matrix.shape[0]
    activation_rates = np.sum(binary_matrix, axis=0) / n_cells
    filtered_matrix = binary_matrix[:, activation_rates >= min_activation_rate]

    # Step 3: Perform TF-IDF transformation
    tfidf_transformer = TfidfTransformer(norm='l2', smooth_idf=True)
    tfidf_matrix = tfidf_transformer.fit_transform(filtered_matrix).toarray()

    return tfidf_matrix

In [ ]:
# preprocessed_data_atac.X = process_scatac_seq(preprocessed_data_atac.X.toarray())

In [ ]:
# print(preprocessed_data_atac.X)

In [29]:
from sklearn.model_selection import train_test_split

train_id, test_id = train_test_split([i for i in range(len(preprocessed_data_rna.obs_names))], test_size=0.1, random_state=42)
train_id, validation_id = train_test_split(train_id, test_size=0.1, random_state=43)

In [30]:
data_rna_true_test = preprocessed_data_rna[test_id]
print(data_rna_true_test)
print("\n")
data_atac_true_test = preprocessed_data_atac[test_id]
print(data_atac_true_test)

View of AnnData object with n_obs × n_vars = 438 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type'
    var: 'gene_ids', 'feature_types', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p'


View of AnnData object with n_obs × n_vars = 438 × 24317
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type'
    var: 'gene_ids', 'feature_types'


## Read reconstructed values

In [3]:
data_rna_pred_test = sc.read_h5ad("/mnt/hdd2/homext/wuch/xn2p/comparison/scButterfly/denovo_try_1_f3000/predict/A2R.h5ad")
print(data_rna_pred_test)

data_atac_pred_test = sc.read_h5ad("/mnt/hdd2/homext/wuch/xn2p/comparison/scButterfly/denovo_try_1_f3000/predict/R2A.h5ad")
print(data_atac_pred_test)

AnnData object with n_obs × n_vars = 438 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type', 'leiden'
    uns: 'cell_type_colors', 'leiden', 'neighbors', 'pca', 'tsne'
    obsm: 'X_pca', 'X_tsne'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
AnnData object with n_obs × n_vars = 438 × 24317
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type', 'leiden'
    uns: 'cell_type_colors', 'leiden', 'neighbors', 'pca', 'tsne'
    obsm: 'X_pca', 'X_tsne'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'


### Run metrics

In [13]:
# 计算 MSE
def metrics_recon(x_true, x_pred):
    if not isinstance(x_pred, np.ndarray):
        x_pred = x_pred.toarray()
    if not isinstance(x_true, np.ndarray):
        x_true = x_true.toarray()
    
    print(f"Shape of x_true: {x_true.shape}")
    # print(np.unique(x_true))
    print(f"Shape of x_pred: {x_pred.shape}")
    # print(np.unique(x_pred))
    print("\n")

    mse_values = calculate_mse(x_true, x_pred)
    print("MSE:", mse_values)

    # r2_ = r2_score(x_true, x_pred)
    # print("R2:", r2_)

    # 计算 PCC
    pcc_values = calculate_pcc(x_true, x_pred)
    print("PCC:", pcc_values)

    # 计算 JSD
    jsd_values = calculate_jsd(x_true, x_pred)
    print("JSD:", jsd_values)

In [14]:
metrics_recon(data_rna_true_test.X, data_rna_pred_test.X)

Shape of x_true: (438, 3000)
Shape of x_pred: (438, 3000)


MSE: 0.05858232453465462
PCC: 0.08804522544324088
JSD: 0.45445735718558233


In [15]:
metrics_recon(data_atac_true_test.X, data_atac_pred_test.X)

Shape of x_true: (438, 24317)
Shape of x_pred: (438, 24317)


MSE: 0.6579751968383789
PCC: 0.043357003
JSD: 0.49863701694183366


## Evaluate clustering

为了定量展示 scButterfly 在跨模态翻译中的优势，我们进一步基于翻译结果的降维数据进行了细胞聚类（使用PCA将翻译图谱的维度降低到50，然后基于降维结果，使用默认分辨率为1的Leiden算法进行细胞聚类），并通过调整互信息 (AMI)、调整兰德指数 (ARI)、同质性得分 (HOM) 和归一化互信息 (NMI) 评估了性能。

In [3]:
import scanpy as sc
import pickle
import numpy as np
import polars as pl
from deeptan.utils.metrics import (
    calculate_jsd,
    AMI,
    ARI,
    NMI,
    kBET,
    ASW,
    AUROC,
    F1,
)

# PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [23]:
data_rna_pred_test = sc.read_h5ad("/mnt/hdd2/homext/wuch/xn2p/comparison/scButterfly/denovo_try_1_f3000/predict/A2R.h5ad")
print(data_rna_pred_test)

data_atac_pred_test = sc.read_h5ad("/mnt/hdd2/homext/wuch/xn2p/comparison/scButterfly/denovo_try_1_f3000/predict/R2A.h5ad")
print(data_atac_pred_test)

AnnData object with n_obs × n_vars = 438 × 3000
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type', 'leiden'
    uns: 'cell_type_colors', 'leiden', 'neighbors', 'pca', 'tsne'
    obsm: 'X_pca', 'X_tsne'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'
AnnData object with n_obs × n_vars = 438 × 24317
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'RNA_snn_res.0.6', 'Cell.type.RNA', 'group', 'cell_type', 'leiden'
    uns: 'cell_type_colors', 'leiden', 'neighbors', 'pca', 'tsne'
    obsm: 'X_pca', 'X_tsne'
    varm: 'PCs'
    obsp: 'connectivities', 'distances'


In [ ]:
adata_pred = sc.concat([data_rna_pred_test, data_atac_pred_test], axis=1)
print(adata_pred)

AnnData object with n_obs × n_vars = 438 × 27317
    varm: 'PCs'


/home/wuch/miniforge3/envs/sc/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


#### Calculate AMI based on Leiden clustering results

In [ ]:
def AMI_adata(adata_, true_label_key="Cell.type.RNA", leiden_key="leiden"):
    sc.pp.pca(adata_)
    sc.pp.neighbors(adata_)
    sc.tl.leiden(adata_)

    unique_categories = list(set(adata_.obs[true_label_key].to_list()))
    category_to_int = {category: idx for idx, category in enumerate(unique_categories)}

    int_categories = [
        category_to_int[category] for category in adata_.obs[true_label_key]
    ]
    ami_ = AMI(int_categories, adata_.obs[leiden_key].astype(int))
    return ami_

In [34]:
AMI_adata(data_rna_pred_test)

np.float64(0.3353013835708497)

In [35]:
AMI_adata(data_rna_true_test)

np.float64(0.4938112492423019)